# Image → Desmos art (v11 · budgeted line art + soft-detail fills)

**What's new vs v10**
- **No more "We only support solved double inequalities" error** – background gradient bands are now one-sided inequalities (`0.94x+0.34y ≥ c` with a box restriction).
- **`TARGET` caps the total expression count (default 4000).** Above ~4k the old budget kept long faint texture and dropped small strong detail → noise. Now every stroke is scored by *edge contrast × √length* and only the best fill the budget; faint, short, wiggly strokes go first.
- **Soft-detail pass (`DETAIL`)** restores low-contrast colour patches the posterizer swallowed (blush, soft shadows): anything still off by ΔE > 3.5 from the flat render becomes an extra polygon.

**Layers (Desmos draws later expressions on top):**
1. background gradient → inequality bands
2. colour regions → polygons + ellipse inequalities + soft-detail patches
3. black line art → Bézier curves, each with its own opacity

## 1 · Setup (silent)

In [ ]:
%%capture
!pip install opencv-python-headless numpy pillow scikit-image matplotlib scipy
print('deps ready')

## 2 · The converter

In [ ]:
%%writefile desmos_mega.py
"""desmos_mega.py v11 -- image -> Desmos art (centerline tracing)

v11 fixes: (1) background bands no longer use unsupported `lo<=expr<=hi`;
(2) --target caps TOTAL expressions and drops the faint/short strokes first;
(3) soft-detail pass (--detail) restores blush / soft shading.


What changed vs v6 (why the output was noisy):
  v6 built a THICK mask (median stroke ~7 px, 19.7% of the image) and traced
  the contours of that mask.  Every thick stroke became a closed loop hugging
  the stroke, which then got fitted as a skinny ellipse / parabola -> the fat
  black dashes and blobs.  Blur edge effects also filled the image border,
  and Sobel/Frangi/k-means layers fired on faint shading.

v7 pipeline:
  1. reflect-pad + non-local-means denoise   (JPEG block noise gone, no border)
  2. single scale-space Canny on luminance    (hysteresis => isolated noise dies)
  3. seal 1 px gaps + skeletonize          -> true 1 px lines
  4. graph trace into OPEN paths, prune spurs, drop short/compact specks
  5. smooth -> cubic-Bezier fit (lines when straight, ellipse for full loops)
  6. expression budget keeps the longest / most important strokes
"""
import os
import json
import time
import argparse
import warnings
from collections import Counter

import cv2
import numpy as np
from scipy.ndimage import gaussian_filter1d
from scipy.spatial import cKDTree

warnings.filterwarnings("ignore")
USE_CONICS = True


# ------------------------------------------------------------------ load
def load_image(image_path, max_size=1600):
    img = cv2.imread(image_path, cv2.IMREAD_UNCHANGED)
    if img is None:
        raise FileNotFoundError(image_path)
    if img.ndim == 2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    if img.shape[2] == 4:
        alpha = img[:, :, 3:4] / 255.0
        img = (img[:, :, :3].astype(float) * alpha
               + 255 * (1 - alpha)).astype(np.uint8)
    h, w = img.shape[:2]
    s = max_size / max(h, w)
    interp = cv2.INTER_CUBIC if s > 1 else cv2.INTER_AREA
    return cv2.resize(img, (int(round(w * s)), int(round(h * s))),
                      interpolation=interp)


# ------------------------------------------------------ 1+2: clean edges
def detect_edges(img, sens=1.5, sigma=1.2, nlm=5, border=4,
                 return_strength=False):
    """Denoised, border-safe, hysteresis Canny on luminance -> bool mask.
    sens < 1 keeps more faint detail, sens > 1 is stricter/cleaner."""
    from skimage.feature import canny
    P = 24                                    # reflect pad: no border artefacts
    pad = cv2.copyMakeBorder(img, P, P, P, P, cv2.BORDER_REFLECT)
    if nlm > 0:
        pad = cv2.fastNlMeansDenoisingColored(pad, None, nlm, nlm, 5, 15)
    L = cv2.cvtColor(pad, cv2.COLOR_BGR2LAB)[..., 0].astype(np.float32) \
        * (100.0 / 255.0)
    e = canny(L, sigma=sigma, low_threshold=0.7 * sens,
              high_threshold=2.0 * sens)
    e = e[P:-P, P:-P]
    mag = None
    if return_strength:                       # edge contrast, L units per px
        from scipy.ndimage import gaussian_gradient_magnitude
        mag = gaussian_gradient_magnitude(L, sigma)[P:-P, P:-P].astype(np.float32)
    if border > 0:                            # kill any residual frame
        e[:border] = e[-border:] = False
        e[:, :border] = e[:, -border:] = False
    return (e, mag) if return_strength else e


# ---------------------------------------------- 3: merge + 1px skeleton
def centerline(edges, merge=0, min_comp=18, seal=False):
    """Edges -> clean 1 px skeleton, speck islands removed.

    seal  : OPTIONAL 3x3 closing for 1-px gaps; off by default (it fuses
            lines that are 2 px apart in dense areas).
    merge : OPTIONAL. >0 fills enclosed gaps up to this many px wide so the
            double edge of a thin stroke becomes one centerline.  Off by
            default: on dense art (eyes, hair) it fuses neighbouring lines
            into blobs whose skeleton is a 'cell wall' mess."""
    from scipy import ndimage as ndi
    from skimage.morphology import skeletonize, remove_small_objects
    m = edges.astype(np.uint8)
    if seal:
        m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, np.ones((3, 3), np.uint8))
    if merge > 0:
        bg = m == 0
        lab, n = ndi.label(bg)                             # 4-connected holes
        if n:
            dt = ndi.distance_transform_edt(bg)
            widest = ndi.maximum(dt, lab, index=np.arange(1, n + 1))
            outer = np.unique(np.concatenate(
                [lab[0], lab[-1], lab[:, 0], lab[:, -1]]))
            ids = np.arange(1, n + 1)
            fill = ids[(widest <= merge / 2.0) & ~np.isin(ids, outer)]
            m[np.isin(lab, fill)] = 1
    sk = skeletonize(m > 0)
    if min_comp > 0:
        sk = remove_small_objects(sk, min_size=min_comp, connectivity=2)
    return sk


# ----------------------------------------- 4: skeleton graph -> paths
_N8 = [(-1, -1), (-1, 0), (-1, 1), (0, 1), (1, 1), (1, 0), (1, -1), (0, -1)]


def _crossing(sk):
    """crossing number per pixel (>=3 => junction), and neighbour count."""
    p = np.pad(sk.astype(np.uint8), 1)
    H, W = sk.shape
    nb = [p[1 + dy:1 + dy + H, 1 + dx:1 + dx + W] for dy, dx in _N8]
    cn = np.zeros((H, W), np.uint8)
    for i in range(8):
        cn += ((nb[i] == 0) & (nb[(i + 1) % 8] == 1)).astype(np.uint8)
    cnt = sum(n.astype(np.uint8) for n in nb)
    return cn, cnt


def _walk(pixels):
    """Order the pixels of one thin component into one or more paths.
    Branches that the crossing-number test misses are walked as extra paths,
    so no skeleton pixel is ever silently dropped."""
    S = set(pixels)
    seen = set()

    def nbrs(q, pool):
        y, x = q
        return [(y + dy, x + dx) for dy, dx in _N8 if (y + dy, x + dx) in pool]

    out = []
    while len(seen) < len(S):
        left = [q for q in pixels if q not in seen]
        pool = set(left)
        start = next((q for q in left if len(nbrs(q, pool)) <= 1), left[0])
        path, cur = [start], start
        seen.add(start)
        while True:
            cand = [n for n in nbrs(cur, S) if n not in seen]
            if not cand:
                break
            cand.sort(key=lambda n: abs(n[0] - cur[0]) + abs(n[1] - cur[1]))
            cur = cand[0]
            path.append(cur)
            seen.add(cur)
        out.append(path)
    return out


def trace_paths(sk, spur=8, rounds=2):
    """Skeleton -> list of ordered paths (arrays of (x, y)); prunes spurs."""
    from scipy import ndimage as ndi
    sk = sk.copy()
    stats = {"spurs_pruned": 0}
    for r in range(rounds + 1):
        cn, cnt = _crossing(sk)
        junc = sk & (cn >= 3)
        seg = sk & ~junc
        lab, n = ndi.label(seg, structure=np.ones((3, 3)))
        ys, xs = np.nonzero(seg)
        order = np.argsort(lab[ys, xs], kind="stable")
        ys, xs, ls = ys[order], xs[order], lab[ys, xs][order]
        cuts = np.flatnonzero(np.diff(ls)) + 1
        jset = set(zip(*np.nonzero(junc)))
        paths, touching = [], []
        for gy, gx in zip(np.split(ys, cuts), np.split(xs, cuts)):
            if len(gy) == 0:
                continue
            for p in _walk(list(zip(gy.tolist(), gx.tolist()))):
                ends_j = []
                for end in (p[0], p[-1]):
                    j = [(end[0] + dy, end[1] + dx) for dy, dx in _N8
                         if (end[0] + dy, end[1] + dx) in jset]
                    ends_j.append(j)
                paths.append((p, ends_j))
        if r == rounds:
            break
        removed = 0                      # prune short branches ending free
        for p, ej in paths:
            free0, free1 = not ej[0], not ej[1]
            if len(p) < spur and (free0 != free1):
                for q in p:
                    sk[q] = False
                removed += 1
        stats["spurs_pruned"] += removed
        if removed == 0:
            break
    out = []
    for p, ej in paths:
        pts = [q for q in p]
        if ej[0]:
            pts = [ej[0][0]] + pts       # reconnect through the junction
        if ej[1]:
            pts = pts + [ej[1][0]]
        a = np.array(pts, float)[:, ::-1]            # (y,x) -> (x,y)
        out.append(a)
    return out, stats


# ------------------------------------------------------- 5: curve fitting
def _is_closed(pts, tol=2.5):
    return len(pts) >= 12 and float(np.hypot(*(pts[0] - pts[-1]))) <= tol


def _path_len(pts):
    return float(np.hypot(*np.diff(pts, axis=0).T).sum())


def smooth_path(pts, sigma, closed):
    if sigma <= 0 or len(pts) < 5:
        return pts
    if closed:
        core = pts[:-1]
        x = gaussian_filter1d(core[:, 0], sigma, mode="wrap")
        y = gaussian_filter1d(core[:, 1], sigma, mode="wrap")
        c = np.stack([x, y], 1)
        return np.vstack([c, c[:1]])
    x = gaussian_filter1d(pts[:, 0], sigma, mode="nearest")
    y = gaussian_filter1d(pts[:, 1], sigma, mode="nearest")
    out = np.stack([x, y], 1)
    out[0], out[-1] = pts[0], pts[-1]     # keep junction attachments exact
    return out


def _try_conic(pts, err):
    """Ellipse only (parabola/hyperbola fits were the v6 noise source)."""
    if not USE_CONICS or len(pts) < 16:
        return None
    cf = _fit_conic_coeffs(pts)
    if cf is None:
        return None
    if cf[0] * cf[2] - cf[1] ** 2 / 4.0 <= 0.05:
        return None
    g = _conic_geom(cf)
    if g is None or g["kind"] != "ellipse":
        return None
    span = float(np.hypot(*(pts.max(0) - pts.min(0))))
    if max(g["a"], g["b"]) > 1.5 * max(span, 1.0):
        return None
    if float(_conic_dists(pts, cf).max()) > err:
        return None
    if _angle_span(pts, g) <= 2.6:
        return None
    g["bbox"] = None
    return g


def _fit_seg(pts, t0, t1, err):
    return _fit_cubic(pts, t0, t1, err)


def fit_path(pts, fit_err, sigma):
    """One ordered pixel path -> list of primitives."""
    closed = _is_closed(pts)
    pts = smooth_path(pts, sigma, closed)
    if len(pts) < 2:
        return []
    chord = pts[-1] - pts[0]
    L = float(np.hypot(*chord))
    if not closed and L > 1e-6:                       # straight? one line
        n = np.array([-chord[1], chord[0]]) / L
        dev = np.abs((pts - pts[0]) @ n).max()
        if dev <= 0.6 * fit_err:
            return [("line", pts[0].copy(), pts[-1].copy())]
    if closed:
        g = _try_conic(pts[:-1], min(fit_err, 1.0))
        if g is not None:
            return [("conic", g)]
        k = len(pts) // 2
        a, b = pts[:k + 1], np.vstack([pts[k:-1], pts[:1]])
        ta = _unit(pts[1] - pts[-2])
        tm = _unit(pts[k - 1] - pts[min(k + 1, len(pts) - 1)])
        return (_fit_seg(a, ta, tm, fit_err)
                + _fit_seg(b, -tm, -ta, fit_err))
    t0 = _unit(pts[min(3, len(pts) - 1)] - pts[0])
    t1 = _unit(pts[max(-4, -len(pts))] - pts[-1])
    return _fit_seg(pts, t0, t1, fit_err)


def paths_to_curves(paths, fit_err=1.2, sigma=1.6, min_len=10):
    """Return list of (length, primitives) per kept path."""
    out = []
    for p in paths:
        ln = _path_len(p)
        if ln < min_len:
            continue
        prims = fit_path(p, fit_err, sigma)
        if prims:
            out.append((ln, prims))
    return out


# ====================== conic section fitting (ellipse) ======================
def _fit_conic_coeffs(pts):
    """General conic via SVD: A x^2 + B xy + C y^2 + D x + E y + F, ||quad||=1."""
    x, y = pts[:, 0], pts[:, 1]
    D = np.column_stack([x * x, x * y, y * y, x, y, np.ones(len(pts))])
    try:
        _, _, Vt = np.linalg.svd(D, full_matrices=False)
    except np.linalg.LinAlgError:
        return None
    cf = Vt[-1]
    n = float(np.linalg.norm(cf[:3]))
    return cf / n if n > 1e-9 else None


def _conic_geom(cf):
    """Classify conic + geometry: kind, center (h,k), axes a>=b, rotation th."""
    A, B, C, D, E, F = cf
    Q = np.array([[A, B / 2], [B / 2, C]])
    Lv = np.array([D, E])
    detQ = A * C - B * B / 4.0
    if abs(detQ) < 1e-9:
        return None
    try:
        center = -0.5 * np.linalg.solve(Q, Lv)
    except np.linalg.LinAlgError:
        return None
    F0 = F + Lv @ center + center @ Q @ center
    if abs(F0) < 1e-12:
        return None
    w, V = np.linalg.eigh(Q)          # ascending eigenvalues
    d1, d0 = -F0 / w[1], -F0 / w[0]
    if detQ > 0:                       # ellipse: both denominators must be > 0
        if d1 <= 0 or d0 <= 0:
            return None                # imaginary ellipse
        a, b = np.sqrt(d1), np.sqrt(d0)
        th = np.arctan2(V[1, 1], V[0, 1])
        kind = "ellipse"
    else:                              # hyperbola: exactly one denominator > 0
        if d1 > 0:
            a, b = np.sqrt(d1), np.sqrt(-d0)
            th = np.arctan2(V[1, 1], V[0, 1])
        else:
            a, b = np.sqrt(d0), np.sqrt(-d1)
            th = np.arctan2(V[1, 0], V[0, 0])
        kind = "hyperbola"
    if not (np.isfinite(a) and np.isfinite(b)) or a <= 0 or b <= 0:
        return None
    return {"kind": kind, "h": float(center[0]), "k": float(center[1]),
            "a": a, "b": b, "th": float(th)}


def _conic_dists(pts, cf):
    A, B, C, D, E, F = cf
    x, y = pts[:, 0], pts[:, 1]
    f = A * x * x + B * x * y + C * y * y + D * x + E * y + F
    gx = 2 * A * x + B * y + D
    gy = B * x + 2 * C * y + E
    return np.abs(f) / np.maximum(np.hypot(gx, gy), 1e-9)


def _arc_bbox(pts, inset=0.25):
    lo = pts.min(0) + inset
    hi = pts.max(0) - inset
    return (float(lo[0]), float(hi[0]), float(lo[1]), float(hi[1]))


def _angle_span(pts, g):
    """Angular coverage of pts around the conic center, in the conic frame."""
    X = pts - np.array([g["h"], g["k"]])
    co, si = np.cos(g["th"]), np.sin(g["th"])
    u = X @ np.array([co, si]) / max(g["a"], 1e-9)
    v = X @ np.array([-si, co]) / max(g["b"], 1e-9)
    ang = np.sort(np.arctan2(v, u))
    gaps = np.diff(np.concatenate([ang, [ang[0] + 2 * np.pi]]))
    return 2 * np.pi - float(gaps.max())


def _finish_conic(pts, g):
    if g["kind"] == "ellipse" and _angle_span(pts, g) > 2.6:
        g["bbox"] = None               # nearly-full ellipse: show it whole
    else:
        g["bbox"] = _arc_bbox(pts)
    return g


# ====================== cubic Bezier curve fitting ======================
def _unit(v):
    n = float(np.hypot(v[0], v[1]))
    return v / n if n > 1e-12 else np.array([1.0, 0.0])


def _chord_u(pts):
    d = np.hypot(np.diff(pts[:, 0]), np.diff(pts[:, 1]))
    u = np.concatenate([[0.0], np.cumsum(d)])
    return u / max(u[-1], 1e-12)


def _bez_eval(P0, P1, P2, P3, u):
    w = 1.0 - u
    return ((w**3)[:, None] * P0 + (3 * w * w * u)[:, None] * P1
            + (3 * w * u * u)[:, None] * P2 + (u**3)[:, None] * P3)


def _bez_d1(P0, P1, P2, P3, u):
    w = 1.0 - u
    return ((3 * w * w)[:, None] * (P1 - P0) + (6 * w * u)[:, None] * (P2 - P1)
            + (3 * u * u)[:, None] * (P3 - P2))


def _generate_bezier(pts, u, t0, t1):
    P0, P3 = pts[0], pts[-1]
    w = 1.0 - u
    a1 = (3 * w * w * u)[:, None] * t0
    a2 = (3 * w * u * u)[:, None] * t1
    c0 = (w * w * (1 + 2 * u))[:, None]       # full coeff of P0 in B(u)
    c3 = (u * u * (3 - 2 * u))[:, None]       # full coeff of P3 in B(u)
    r = pts - c0 * P0 - c3 * P3
    C00, C01 = float((a1 * a1).sum()), float((a1 * a2).sum())
    C11 = float((a2 * a2).sum())
    X0, X1 = float((a1 * r).sum()), float((a2 * r).sum())
    det = C00 * C11 - C01 * C01
    L = float(np.hypot(*(P3 - P0)))
    eps = 1e-6 * max(L, 1e-9)
    fallback = max(L / 3.0, eps)
    if abs(det) < 1e-12 * max(C00 * C11, 1e-18):
        al = ar = fallback
    else:
        al = (X0 * C11 - X1 * C01) / det
        ar = (C00 * X1 - C01 * X0) / det
        bad = (not np.isfinite(al)) or (not np.isfinite(ar)) \
            or al < eps or ar < eps or al > 4 * L or ar > 4 * L
        if bad:
            al = ar = fallback
    return P0, P0 + al * t0, P3 + ar * t1, P3


def _reparam(pts, bez, u):
    P0, P1, P2, P3 = bez
    for _ in range(4):
        q = _bez_eval(P0, P1, P2, P3, u) - pts
        q1 = _bez_d1(P0, P1, P2, P3, u)
        step = (q * q1).sum(1) / np.maximum((q1 * q1).sum(1), 1e-12)
        u = np.clip(u - step, 1e-4, 1.0 - 1e-4)
    return u


def _max_error(pts, bez, u):
    q = _bez_eval(*bez, u) - pts
    dist = np.hypot(q[:, 0], q[:, 1])
    i = int(np.argmax(dist))
    return float(dist[i]), i


def _center_tangent(pts, i):
    return _unit(pts[i - 1] - pts[min(i + 1, len(pts) - 1)])


def _fit_cubic(pts, t0, t1, err):
    n = len(pts)
    if n == 2:
        return [("line", pts[0], pts[1])]
    u = _chord_u(pts)
    bez = _generate_bezier(pts, u, t0, t1)
    e, i = _max_error(pts, bez, u)
    if e > err:
        u = _reparam(pts, bez, u)
        bez = _generate_bezier(pts, u, t0, t1)
        e, i = _max_error(pts, bez, u)
    if e <= err:
        return [bez]
    if i < 1 or i > n - 2:
        i = n // 2
    ct = _center_tangent(pts, i)
    return (_fit_seg(pts[:i + 1], t0, ct, err)
            + _fit_seg(pts[i:], -ct, t1, err))



# ------------------------------------------------------- coords + latex
def _to_math(p, h, w, scale):
    return ((p[0] - w / 2.0) * scale, (h / 2.0 - p[1]) * scale)


def curves_to_math(curves, h, w, scale=1.0):
    out = []
    for c in curves:
        if isinstance(c[0], str):
            if c[0] == "line":
                out.append(("line", _to_math(c[1], h, w, scale),
                            _to_math(c[2], h, w, scale)))
            else:  # conic
                g = dict(c[1])
                g["th"] = -g["th"]   # pixel y-down -> math y-up
                g["h"], g["k"] = _to_math((g["h"], g["k"]), h, w, scale)
                g["a"] *= scale
                if np.isfinite(g.get("b", float("nan"))):
                    g["b"] *= scale
                bb = g.get("bbox")
                if bb is not None:
                    g["bbox"] = ((bb[0] - w / 2.0) * scale,
                                 (bb[1] - w / 2.0) * scale,
                                 (h / 2.0 - bb[3]) * scale,
                                 (h / 2.0 - bb[2]) * scale)
                out.append(("conic", g))
        else:
            out.append(tuple(_to_math(p, h, w, scale) for p in c))
    return out


def _fmt(v):
    if abs(v) < 5e-5:
        v = 0.0
    return f"{v:.4f}"


def _dx(var, c):
    if abs(c) < 5e-5:
        return var
    return f"{var}-{_fmt(c)}" if c > 0 else f"{var}+{_fmt(-c)}"


def _line_latex(a, b):
    x1, y1, x2, y2 = a[0], a[1], b[0], b[1]
    dx, dy = x2 - x1, y2 - y1
    if abs(dx) < 1e-6:
        lo, hi = sorted((y1, y2))
        return f"x={_fmt(x1)}\\left\\{{{_fmt(lo)}<y<{_fmt(hi)}\\right\\}}"
    m, bb = dy / dx, y1 - dy / dx * x1
    lo, hi = sorted((x1, x2))
    return (f"y={_fmt(m)}x{'+' if bb >= 0 else '-'}{_fmt(abs(bb))}"
            f"\\left\\{{{_fmt(lo)}<x<{_fmt(hi)}\\right\\}}")


def _poly(c3, c2, c1, c0):
    ts = [f"{_fmt(c3)}t^{{3}}", f"{_fmt(c2)}t^{{2}}",
          f"{_fmt(c1)}t", _fmt(c0)]
    s = ts[0]
    for t in ts[1:]:
        s += "+" + t if not t.startswith("-") else t
    return s


def _bez_is_line(bez, tol=0.6):
    P0, P1, P2, P3 = [np.asarray(p, float) for p in bez]
    v = P3 - P0
    L = float(np.hypot(v[0], v[1]))
    if L < 1e-9:
        return True
    n = np.array([-v[1], v[0]]) / L
    dev = max(abs(float(np.dot(P1 - P0, n))), abs(float(np.dot(P2 - P0, n))))
    return dev < tol


def _curve_latex(bez):
    P0, P1, P2, P3 = [np.asarray(p, float) for p in bez]
    x = (P3[0] - 3 * P2[0] + 3 * P1[0] - P0[0],
         3 * P2[0] - 6 * P1[0] + 3 * P0[0],
         3 * P1[0] - 3 * P0[0], P0[0])
    y = (P3[1] - 3 * P2[1] + 3 * P1[1] - P0[1],
         3 * P2[1] - 6 * P1[1] + 3 * P0[1],
         3 * P1[1] - 3 * P0[1], P0[1])
    return (f"\\left({_poly(*x)},\\ {_poly(*y)}\\right)"
            f"\\left\\{{0\\le t\\le1\\right\\}}")


def _with_bbox(s, bbox):
    if bbox is None:
        return s
    x0, x1, y0, y1 = bbox
    return (s + f"\\left\\{{{_fmt(x0)}<x<{_fmt(x1)},\\ "
                 f"{_fmt(y0)}<y<{_fmt(y1)}\\right\\}}")


def _conic_latex(g, rel="="):
    """Desmos LaTeX for an ellipse/circle in math coords.
    rel '=' draws the outline, '\\le' shades the filled interior (inequality)."""
    h, k, a, b, th = g["h"], g["k"], g["a"], g["b"], g["th"]
    X = f"\\left({_dx('x', h)}\\right)"
    Y = f"\\left({_dx('y', k)}\\right)"
    if abs(a - b) <= 1e-3 * max(a, b):
        return f"{X}^{{2}}+{Y}^{{2}}{rel}{_fmt(a * a)}"
    if abs(np.sin(th)) < 0.01:
        return (f"\\frac{{{X}^{{2}}}}{{{_fmt(a * a)}}}+"
                f"\\frac{{{Y}^{{2}}}}{{{_fmt(b * b)}}}{rel}1")
    if abs(np.cos(th)) < 0.01:
        return (f"\\frac{{{Y}^{{2}}}}{{{_fmt(a * a)}}}+"
                f"\\frac{{{X}^{{2}}}}{{{_fmt(b * b)}}}{rel}1")
    T = _fmt(th)                       # the ANGLE goes inside cos/sin
    u_ = (f"\\left({X}\\cos\\left({T}\\right)"
          f"+{Y}\\sin\\left({T}\\right)\\right)")
    v_ = (f"\\left(-{X}\\sin\\left({T}\\right)"
          f"+{Y}\\cos\\left({T}\\right)\\right)")
    return (f"\\frac{{{u_}^{{2}}}}{{{_fmt(a * a)}}}+"
            f"\\frac{{{v_}^{{2}}}}{{{_fmt(b * b)}}}{rel}1")


def curves_to_desmos(curves):
    out, seen = [], set()
    for c in curves:
        if isinstance(c[0], str):
            if c[0] == "line":
                s = _line_latex(c[1], c[2])
            else:
                s = _with_bbox(_conic_latex(c[1]), c[1].get("bbox"))
        elif _bez_is_line(c):
            s = _line_latex(c[0], c[3])
        else:
            s = _curve_latex(c)
        if s not in seen:
            seen.add(s)
            out.append(s)
    return out


# ------------------------------------------------------- save
def save_outputs(lines, output_base, width, height, scale):
    vw = round(width / 2.0 * scale * 1.05, 2)
    vh = round(height / 2.0 * scale * 1.05, 2)
    state = {"version": 9, "graph": {
        "viewport": {"xmin": -vw, "ymin": -vh, "xmax": vw, "ymax": vh},
        "expressions": {"list": [
            {"type": "expression", "id": f"e{i+1}", "latex": ln,
             "color": "#000000", "lines": True, "points": False}
            for i, ln in enumerate(lines)]}}}
    txt, js = output_base + ".txt", output_base + ".json"
    with open(txt, "w", encoding="utf-8") as f:
        f.write("\n".join(lines) + "\n")
    with open(js, "w", encoding="utf-8") as f:
        json.dump(state, f)
    return txt, js


# ====================== color fills: polygons + inequalities ======================
def posterize(img, K=14, nlm=6, sample=300000, seed=0):
    """Denoise, then k-means in Lab -> (denoised BGR image, label map)."""
    den = (cv2.fastNlMeansDenoisingColored(img, None, nlm, nlm, 5, 15)
           if nlm > 0 else img.copy())
    lab = cv2.cvtColor(den, cv2.COLOR_BGR2LAB).astype(np.float32)
    h, w = lab.shape[:2]
    Z = lab.reshape(-1, 3)
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(Z), min(sample, len(Z)), replace=False)
    cv2.setRNGSeed(seed)                       # reproducible k-means
    crit = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.5)
    _, _, cent = cv2.kmeans(Z[idx], K, None, crit, 2, cv2.KMEANS_PP_CENTERS)
    out = np.empty(len(Z), np.int32)
    for i in range(0, len(Z), 400000):
        d = ((Z[i:i + 400000, None, :] - cent[None]) ** 2).sum(2)
        out[i:i + 400000] = d.argmin(1)
    return den, out.reshape(h, w)


def smooth_labels(labels, K, sigma=1.5):
    """Soft majority vote: removes speckle, rounds jagged region borders."""
    best = np.full(labels.shape, -1.0, np.float32)
    out = np.zeros(labels.shape, np.int32)
    for k in range(K):
        f = cv2.GaussianBlur((labels == k).astype(np.float32), (0, 0), sigma)
        upd = f > best
        out[upd] = k
        best[upd] = f[upd]
    return out


def merge_small(labels, K, min_area, rounds=2):
    """Re-label components smaller than min_area with their neighbour's label."""
    H, W = labels.shape
    for _ in range(rounds):
        changed = 0
        for k in range(K):
            m = (labels == k).astype(np.uint8)
            n, cc, stats, _ = cv2.connectedComponentsWithStats(m, connectivity=4)
            for i in range(1, n):
                if stats[i, cv2.CC_STAT_AREA] >= min_area:
                    continue
                x, y, w, h = stats[i, :4]
                x0, y0 = max(x - 2, 0), max(y - 2, 0)
                x1, y1 = min(x + w + 2, W), min(y + h + 2, H)
                sub = cc[y0:y1, x0:x1] == i
                ring = cv2.dilate(sub.astype(np.uint8), np.ones((3, 3), np.uint8)) > 0
                nb = labels[y0:y1, x0:x1][ring & ~sub]
                nb = nb[nb != k]
                if len(nb):
                    labels[y0:y1, x0:x1][sub] = np.bincount(nb).argmax()
                    changed += 1
        if not changed:
            break
    return labels


def _hex(bgr):
    b, g, r = [int(round(float(v))) for v in bgr]
    return "#%02x%02x%02x" % (max(0, min(255, r)), max(0, min(255, g)),
                              max(0, min(255, b)))


def find_background(labels, min_frac=0.02):
    """Label that dominates the image border + its border-touching pieces."""
    H, W = labels.shape
    border = np.concatenate([labels[0], labels[-1], labels[:, 0], labels[:, -1]])
    k = int(np.bincount(border).argmax())
    n, cc, stats, _ = cv2.connectedComponentsWithStats(
        (labels == k).astype(np.uint8), connectivity=4)
    edge = set(np.unique(np.concatenate([cc[0], cc[-1], cc[:, 0], cc[:, -1]]))) - {0}
    keep = np.zeros(n, bool)
    for i in edge:
        if stats[i, cv2.CC_STAT_AREA] >= min_frac * H * W:
            keep[i] = True
    return k, keep[cc]


def fit_background(den, mask, step=1.0, max_bands=40):
    """Linear colour gradient over the background -> list of inequality bands.
    Each band is  lo <= nx*x + ny*y <= hi  (math coords) with its own colour."""
    H, W = mask.shape
    m = cv2.erode(mask.astype(np.uint8), np.ones((9, 9), np.uint8)) > 0
    if m.sum() < 500:
        m = mask
    ys, xs = np.nonzero(m)
    if len(ys) > 200000:
        sel = np.random.default_rng(1).choice(len(ys), 200000, replace=False)
        ys, xs = ys[sel], xs[sel]
    X, Y = xs - W / 2.0, H / 2.0 - ys
    A = np.stack([np.ones(len(X)), X, Y], 1)
    rgb = den[ys, xs][:, ::-1].astype(float)
    coef = np.linalg.lstsq(A, rgb, rcond=None)[0]            # (3 rows, 3 channels)
    c0, gx, gy = coef[0], coef[1], coef[2]
    wl = np.array([0.299, 0.587, 0.114])
    g = np.array([gx @ wl, gy @ wl])
    mag = float(np.hypot(*g))
    mean_bgr = rgb.mean(0)[::-1]
    n = g / mag if mag > 1e-9 else np.array([0.0, 1.0])
    corners = np.array([[sx * W / 2.0, sy * H / 2.0] for sx in (-1, 1) for sy in (-1, 1)])
    t = corners @ n
    t0, t1 = float(t.min()), float(t.max())
    chan_slope = np.abs(np.stack([gx, gy], 1) @ n)           # levels per px, per channel
    delta = float(chan_slope.max() * (t1 - t0))
    nb = int(np.clip(np.ceil(delta / step), 1, max_bands))
    if nb <= 1:
        return [dict(kind="band", flat=True, color=_hex(mean_bgr))], {
            "bg_gradient_levels": round(delta, 1), "bg_bands": 1}
    edges = np.linspace(t0, t1, nb + 1)
    bands = []
    for i in range(nb):
        tm = 0.5 * (edges[i] + edges[i + 1])
        col = c0 + (gx * n[0] + gy * n[1]) * tm              # plane value on the axis
        hexc = _hex(col[::-1])
        lo, hi = float(edges[i]), float(edges[i + 1]) + (1.0 if i < nb - 1 else 2.0)
        if i == 0:
            lo -= 2.0
        if bands and bands[-1]["color"] == hexc:
            bands[-1]["hi"] = hi
        else:
            bands.append(dict(kind="band", flat=False, n=(float(n[0]), float(n[1])),
                              lo=lo, hi=hi, color=hexc))
    return bands, {"bg_gradient_levels": round(delta, 1), "bg_bands": len(bands)}


def _closed_contour(c, x, y, smooth, eps, max_pts):
    pts = c.reshape(-1, 2).astype(float) + np.array([x, y], float)
    if len(pts) < 3:
        return None
    pts = smooth_path(np.vstack([pts, pts[:1]]), smooth, True)[:-1] if smooth > 0 else pts
    e = eps
    for _ in range(8):
        s = cv2.approxPolyDP(pts.astype(np.float32).reshape(-1, 1, 2), e, True).reshape(-1, 2)
        if len(s) <= max_pts:
            break
        e *= 1.4
    return s.astype(float) if len(s) >= 3 else None


def build_color_layers(img, K=14, min_area=150, smooth=1.5, eps=1.1, max_pts=400,
                       max_ellipses=12, ellipse_iou=0.93, nlm=6, use_bg=True,
                       band_step=1.0, max_polys=0, detail=3.5, max_detail=150,
                       verbose=True):
    """Image -> painter-ordered layers (bands, polygons, ellipse inequalities)."""
    T = time.time()
    H, W = img.shape[:2]
    log = print if verbose else (lambda *a, **k: None)
    den, labels = posterize(img, K, nlm)
    K = int(labels.max()) + 1
    labels = smooth_labels(labels, K, smooth)
    labels = merge_small(labels, K, min_area)
    rep = {"colors_K": K}
    bands, bgmask = [], None
    if use_bg:
        bl, bgmask = find_background(labels)
        if bgmask.any():
            bands, brep = fit_background(den, bgmask, band_step)
            rep.update(brep)
            rep["bg_pixels_%"] = round(100.0 * float(bgmask.mean()), 1)
            labels = labels.copy()
            labels[bgmask] = -1                       # drawn by the inequality bands
    log(f"      colors     : K={K}, background -> {len(bands)} inequality band(s)"
        if bands else f"      colors     : K={K}, no background band")

    regs = []
    for k in range(K):
        m = (labels == k).astype(np.uint8)
        n, cc, stats, _ = cv2.connectedComponentsWithStats(m, connectivity=4)
        for i in range(1, n):
            x, y, w, h, a = [int(v) for v in stats[i]]
            if a < min_area:
                continue
            sub = (cc[y:y + h, x:x + w] == i).astype(np.uint8)
            cs, _ = cv2.findContours(sub, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
            if not cs:
                continue
            c = max(cs, key=cv2.contourArea)
            fm = np.zeros_like(sub)
            cv2.drawContours(fm, [c], -1, 1, -1)
            fpx = int(fm.sum())
            col = den[y:y + h, x:x + w][sub > 0].mean(0)
            r = dict(kind="poly", color=_hex(col), filled=fpx, area=a)
            if len(c) >= 5 and fpx >= 40:             # try an exact ellipse
                (cx, cy), (d1, d2), ang = cv2.fitEllipse(c)
                em = np.zeros_like(sub)
                cv2.ellipse(em, (int(round(cx * 8)), int(round(cy * 8))),
                            (int(round(d1 * 4)), int(round(d2 * 4))), ang, 0, 360, 1, -1,
                            cv2.LINE_8, 3)
                iou = float((em & fm).sum()) / max(float((em | fm).sum()), 1.0)
                if iou >= ellipse_iou and max(d1, d2) < 0.5 * max(W, H):
                    th = np.radians(ang) + (np.pi / 2 if d2 > d1 else 0.0)
                    r["ell"] = dict(h=cx + x, k=cy + y, a=max(d1, d2) / 2.0,
                                    b=min(d1, d2) / 2.0, th=float(th), kind="ellipse")
                    r["iou"] = iou
            r["pts"] = _closed_contour(c, x, y, smooth, eps, max_pts)
            if r["pts"] is None and "ell" not in r:
                continue
            regs.append(r)
    ell_ok = sorted([r for r in regs if "ell" in r], key=lambda r: -r["filled"])
    chosen = set(id(r) for r in ell_ok[:max_ellipses])
    layers = list(bands)
    for r in sorted(regs, key=lambda r: -r["filled"]):
        if id(r) in chosen:
            layers.append(dict(kind="ellipse", color=r["color"], ell=r["ell"],
                               filled=r["filled"], iou=r["iou"]))
        elif r["pts"] is not None:
            layers.append(dict(kind="poly", color=r["color"], pts=r["pts"],
                               filled=r["filled"]))
    if max_polys and sum(L["kind"] == "poly" for L in layers) > max_polys:
        # fill budget: keep the biggest regions (they carry the picture);
        # tiny ones are what turned into noise
        polys_all = sorted((L for L in layers if L["kind"] == "poly"),
                           key=lambda L: -L["filled"])
        drop = set(id(L) for L in polys_all[max_polys:])
        rep["polygons_dropped_budget"] = len(drop)
        layers = [L for L in layers if id(L) not in drop]
    if detail and detail > 0:
        flat0 = render_color(layers, H, W)
        extra = []
        for rnd in range(2):                      # 2nd round catches what is left
            left = max_detail - len(extra)            # cap is TOTAL, not per round
            if left <= 0:
                break
            ex = residual_layers(den, flat0, thr=detail, max_n=left)
            if not ex:
                break
            extra += ex
            flat0 = render_color(layers + extra, H, W)
        layers = layers + extra
        rep["detail_patches"] = len(extra)
        log(f"      detail     : +{len(extra)} soft patches (blush / shading), "
            f"delta-E > {detail}")
    polys = [L for L in layers if L["kind"] == "poly"]
    vs = np.array([len(L["pts"]) for L in polys]) if polys else np.array([0])
    rep.update({"polygons": len(polys),
                "ellipse_inequalities": sum(L["kind"] == "ellipse" for L in layers),
                "polygon_vertices": {"total": int(vs.sum()), "median": int(np.median(vs)),
                                     "max": int(vs.max())},
                "fill_layers": len(layers), "seconds": round(time.time() - T, 1)})
    flat = render_color(layers, H, W)
    err = np.abs(flat.astype(float) - den.astype(float))
    mse = float(((flat.astype(float) - den.astype(float)) ** 2).mean())
    rep["color_fidelity"] = {
        "mean_abs_error_levels": round(float(err.mean()), 2),
        "psnr_db": round(10 * np.log10(255.0 ** 2 / max(mse, 1e-9)), 1),
        "pixels_within_16_levels_%": round(100.0 * float((err.max(2) <= 16).mean()), 1)}
    log(f"      fills      : {len(polys)} polygons + {rep['ellipse_inequalities']} ellipse "
        f"inequalities + {len(bands)} bands = {len(layers)} layers")
    log(f"      color fid. : {rep['color_fidelity']}")
    return layers, rep, {"labels": labels, "flat": flat, "den": den}


def residual_layers(den, flat, thr=5.0, min_area=40, max_n=150, smooth=1.2,
                    eps=0.9, max_pts=120):
    """Colour patches the K-means posterise swallowed (blush, soft shadows).

    Compares the denoised source with the flat render in true Lab (delta-E);
    blobs where they still differ by > thr and are bigger than a thin edge
    sliver become extra polygons painted on top of the base fills."""
    H, W = flat.shape[:2]
    a = cv2.cvtColor(cv2.GaussianBlur(den, (0, 0), 1.2).astype(np.float32) / 255.0,
                     cv2.COLOR_BGR2Lab)
    b = cv2.cvtColor(flat.astype(np.float32) / 255.0, cv2.COLOR_BGR2Lab)
    dE = np.sqrt(((a - b) ** 2).sum(2))
    m = (dE > thr).astype(np.uint8)
    # ignore a band around the flat render's own colour borders: there the
    # difference is just anti-aliasing / slightly shifted edges, not detail
    bd = np.zeros((H, W), np.uint8)
    f16 = flat.astype(np.int16)
    bd[:, 1:] |= (np.abs(f16[:, 1:] - f16[:, :-1]).sum(2) > 0)
    bd[1:, :] |= (np.abs(f16[1:, :] - f16[:-1, :]).sum(2) > 0)
    bd = cv2.dilate(bd, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9)))
    m[bd > 0] = 0
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    m = cv2.morphologyEx(m, cv2.MORPH_OPEN, k)          # drop border slivers
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, k)
    n, cc, stats, _ = cv2.connectedComponentsWithStats(m, connectivity=4)
    cand = []
    for i in range(1, n):
        x, y, w_, h_, ar = [int(v) for v in stats[i]]
        if ar < min_area or ar > 0.25 * H * W:
            continue
        sub = (cc[y:y + h_, x:x + w_] == i).astype(np.uint8)
        cs, _ = cv2.findContours(sub, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        if not cs:
            continue
        c = max(cs, key=cv2.contourArea)
        pts = _closed_contour(c, x, y, smooth, eps, max_pts)
        if pts is None:
            continue
        sel = sub > 0
        col = np.median(den[y:y + h_, x:x + w_][sel], axis=0)
        score = float(dE[y:y + h_, x:x + w_][sel].mean()) * np.sqrt(ar)
        cand.append((score, ar, dict(kind="poly", color=_hex(col), pts=pts,
                                     filled=int(ar))))
    cand.sort(key=lambda t: -t[0])
    cand = cand[:max_n]
    cand.sort(key=lambda t: -t[1])                      # big first, small on top
    return [c[2] for c in cand]


def render_color(layers, h, w):
    """Painter's-order raster of the fill layers (first = bottom)."""
    canvas = np.zeros((h, w, 3), np.uint8)
    X = (np.arange(w, dtype=np.float32) - w / 2.0)[None, :]
    Y = (h / 2.0 - np.arange(h, dtype=np.float32))[:, None]
    for L in layers:
        rgb = L["color"].lstrip("#")
        bgr = (int(rgb[4:6], 16), int(rgb[2:4], 16), int(rgb[0:2], 16))
        if L["kind"] == "band":
            if L["flat"]:
                canvas[:] = bgr
            else:
                t = L["n"][0] * X + L["n"][1] * Y
                canvas[t >= L["lo"]] = bgr
        elif L["kind"] == "poly":
            cv2.fillPoly(canvas, [np.round(L["pts"] * 16).astype(np.int32)],
                         bgr, cv2.LINE_8, 4)
        else:
            e = L["ell"]
            cv2.ellipse(canvas, (int(round(e["h"] * 8)), int(round(e["k"] * 8))),
                        (int(round(e["a"] * 8)), int(round(e["b"] * 8))),
                        float(np.degrees(e["th"])), 0, 360, bgr, -1, cv2.LINE_8, 3)
    return canvas


def fills_to_expressions(layers, h, w, scale=1.0):
    """Layers -> list of dict(latex, color, kind) in Desmos math coordinates."""
    hw, hh = w / 2.0 * scale, h / 2.0 * scale
    box = (f"\\left\\{{\\left|x\\right|\\le{_fmt(hw)}\\right\\}}"
           f"\\left\\{{\\left|y\\right|\\le{_fmt(hh)}\\right\\}}")
    out = []
    for L in layers:
        if L["kind"] == "band":
            if L["flat"]:
                s = f"\\left|x\\right|\\le{_fmt(hw)}\\left\\{{\\left|y\\right|\\le{_fmt(hh)}\\right\\}}"
            else:
                nx, ny = L["n"]
                # Desmos only accepts *solved* double inequalities (a<y<b), so
                # `lo <= nx*x+ny*y <= hi` is rejected.  Bands are painted in
                # ascending order, so each one only needs its lower bound:
                # later bands simply cover the upper part of the earlier one.
                s = (f"{_fmt(nx)}x+{_fmt(ny)}y\\ge{_fmt(L['lo'] * scale)}"
                     f"{box}").replace("+-", "-")
            out.append(dict(latex=s, color=L["color"], kind="band"))
        elif L["kind"] == "poly":
            p = L["pts"]
            xy = np.stack([(p[:, 0] - w / 2.0) * scale, (h / 2.0 - p[:, 1]) * scale], 1)
            xy = np.round(xy, 1)
            keep = np.ones(len(xy), bool)
            keep[1:] = np.any(np.diff(xy, axis=0) != 0, axis=1)
            xy = xy[keep]
            if len(xy) < 3:
                continue
            pts = ",".join(f"\\left({a:.1f},{b:.1f}\\right)" for a, b in xy)
            out.append(dict(latex=f"\\operatorname{{polygon}}\\left({pts}\\right)",
                            color=L["color"], kind="poly"))
        else:
            e = L["ell"]
            g = dict(kind="ellipse", h=(e["h"] - w / 2.0) * scale,
                     k=(h / 2.0 - e["k"]) * scale, a=e["a"] * scale, b=e["b"] * scale,
                     th=-e["th"])                       # y flips -> angle flips
            out.append(dict(latex=_conic_latex(g, "\\le"), color=L["color"],
                            kind="ellipse"))
    return out


def save_color_outputs(fills, lines, output_base, width, height, scale,
                       line_width="1.5", folders=True, line_opacity=None):
    """Desmos state: folder 1 background bands, folder 2 colour fills, folder 3
    BLACK line art on top.  (Later expressions are drawn on top in Desmos.)"""
    vw = round(width / 2.0 * scale * 1.05, 2)
    vh = round(height / 2.0 * scale * 1.05, 2)
    bg = [f for f in fills if f["kind"] == "band"]
    fg = [f for f in fills if f["kind"] != "band"]
    ex, n = [], 0

    def folder(fid, title):
        if folders:
            ex.append({"type": "folder", "id": fid, "title": title, "collapsed": True})

    def fill_expr(f, fid):
        nonlocal n
        n += 1
        d = {"type": "expression", "id": f"e{n}", "latex": f["latex"],
             "color": f["color"], "lines": True, "lineWidth": "1",
             "lineOpacity": "1", "fillOpacity": "1", "points": False}
        if folders:
            d["folderId"] = fid
        ex.append(d)

    if bg:
        folder("f_bg", "1 - background gradient (inequalities)")
        for f in bg:
            fill_expr(f, "f_bg")
    folder("f_fill", "2 - color regions (polygons + ellipse inequalities)")
    for f in fg:
        fill_expr(f, "f_fill")
    folder("f_line", "3 - line art (black)")
    for li, ln in enumerate(lines):
        n += 1
        d = {"type": "expression", "id": f"e{n}", "latex": ln, "color": "#000000",
             "lines": True, "points": False, "lineWidth": line_width,
             "lineOpacity": (f"{line_opacity[li]:.2f}"
                             if line_opacity is not None else "1")}
        if folders:
            d["folderId"] = "f_line"
        ex.append(d)
    state = {"version": 9, "graph": {
        "viewport": {"xmin": -vw, "ymin": -vh, "xmax": vw, "ymax": vh},
        "expressions": {"list": ex}}}
    txt, js = output_base + ".txt", output_base + ".json"
    with open(txt, "w", encoding="utf-8") as f:
        f.write("\n".join([x["latex"] for x in bg + fg] + lines) + "\n")
    with open(js, "w", encoding="utf-8") as f:
        json.dump(state, f)
    extra = write_paste_workflow(ex, output_base)
    return txt, js, n, extra


_JS_HEAD = r"""// ===== Desmos style applier (generated) =====
// 1) Paste the equations from the .txt into an EMPTY Desmos graph
//    (one expression per line, nothing else in the graph).
// 2) Paste this whole script into the browser console (F12) + Enter.
window.styleById = {};
"""

_JS_TAIL = r"""(function () {
  const EXPECTED = __EXPECTED__;
  const styleList = Object.keys(window.styleById)
    .sort((a, b) => parseInt(a.slice(1)) - parseInt(b.slice(1)))
    .map(k => window.styleById[k]);
  const state = Calc.getState();
  // empty expressions (Desmos keeps a blank one at the end) are ignored
  const live = state.expressions.list.filter(
    e => e.type === "expression" && e.latex && String(e.latex).trim() !== "");
  console.log("styles:", styleList.length, "| non-empty expressions in graph:", live.length);
  if (live.length !== EXPECTED && !window.FORCE_APPLY) {
    console.warn("The graph has " + live.length + " expressions but " + EXPECTED +
      " are expected (start from an EMPTY graph and make sure every line pasted)." +
      " Styles NOT applied, because they would land on the wrong expressions." +
      " To apply anyway: window.FORCE_APPLY = true, then re-run.");
    return;
  }
  let applied = 0;
  live.forEach((e, i) => {
    const s = styleList[i];
    if (!s) return;
    ["color", "fillOpacity", "lineOpacity", "lineWidth"].forEach(k => {
      if (s[k] !== undefined) e[k] = s[k];
    });
    applied++;
  });
  Calc.setState(state);
  console.log("Applied styles to", applied, "expressions");
})();
"""


def write_paste_workflow(ex, output_base, chunk=500):
    """For the 'paste equations, then apply styles from the console' workflow.

    <base>.txt                 equations, one per line, in draw order
    <base>_style_K.txt         {"e1":{color,fillOpacity,lineOpacity,lineWidth},...}
                               (<= `chunk` entries each)
    <base>_apply_styles.js     the same styles embedded in one ready console script
    Style ids e1..eN follow the line order of <base>.txt."""
    style = {}
    for e in ex:
        if e.get("type") != "expression":
            continue
        style[e["id"]] = {k: e[k] for k in ("color", "fillOpacity", "lineOpacity",
                                            "lineWidth") if k in e}
    items = list(style.items())
    chunks = [json.dumps(dict(items[i:i + chunk]), separators=(",", ":"))
              for i in range(0, len(items), chunk)]
    names = []
    for k, c in enumerate(chunks, 1):
        name = f"{output_base}_style_{k}.txt"
        with open(name, "w", encoding="utf-8") as f:
            f.write(c)
        names.append(name)
    body = "".join(f"Object.assign(window.styleById, JSON.parse(`{c}`));\n"
                   for c in chunks)
    js = output_base + "_apply_styles.js"
    with open(js, "w", encoding="utf-8") as f:
        f.write(_JS_HEAD + body + _JS_TAIL.replace("__EXPECTED__", str(len(items))))
    return names, js


def curve_strength(curves, mag):
    """Mean edge contrast (gradient magnitude of the source) along each curve."""
    from scipy.ndimage import map_coordinates
    out = []
    for c in curves:
        pts = np.vstack(_sample(c, 1.0))
        v = map_coordinates(mag, [pts[:, 1], pts[:, 0]], order=1, mode="nearest")
        out.append(float(v.mean()))
    return out


def line_opacities(strength, lo=0.30, hi=0.90, gamma=0.8, ref_pct=90):
    """Edge contrast -> per-curve opacity in [lo, hi].
    Contrast at the ref_pct percentile (and above) gets `hi`; faint edges fade
    towards `lo`.  gamma < 1 lifts mid-contrast lines a little."""
    s = np.asarray(strength, float)
    if s.size == 0:
        return []
    ref = max(float(np.percentile(s, ref_pct)), 1e-6)
    t = np.clip(s / ref, 0.0, 1.0) ** gamma
    return [round(float(v), 2) for v in lo + (hi - lo) * t]


def curves_to_desmos_op(curves, opac):
    """Like curves_to_desmos, but keeps a per-line opacity list aligned with the
    output (duplicates keep the stronger opacity)."""
    out, ops, seen = [], [], {}
    for c, o in zip(curves, opac):
        if isinstance(c[0], str):
            if c[0] == "line":
                s = _line_latex(c[1], c[2])
            else:
                s = _with_bbox(_conic_latex(c[1]), c[1].get("bbox"))
        elif _bez_is_line(c):
            s = _line_latex(c[0], c[3])
        else:
            s = _curve_latex(c)
        if s in seen:
            ops[seen[s]] = max(ops[seen[s]], o)
        else:
            seen[s] = len(out)
            out.append(s)
            ops.append(o)
    return out, ops


def render_ink(curves, opac, h, w, levels=8, ss=2):
    """Ink density 0..1 of black lines with per-curve opacity (alpha-composited)."""
    opac = np.asarray(opac, float)
    ink = np.zeros((h, w), np.float32)
    if len(opac) == 0:
        return ink
    span = float(np.ptp(opac))
    q = (np.zeros(len(opac), int) if span < 1e-6 else
         np.clip(np.round((opac - opac.min()) / span * (levels - 1)).astype(int), 0, levels - 1))
    for lv in range(levels):
        idx = np.flatnonzero(q == lv)
        if len(idx) == 0:
            continue
        a = float(opac[idx].mean())
        cov = 1.0 - render_curves([curves[i] for i in idx], h, w, ss=ss).astype(np.float32) / 255.0
        ink = 1.0 - (1.0 - ink) * (1.0 - a * cov)
    return ink


def compose_preview(flat, line_img, ink=None):
    """Colour fills with the black line art on top.  `ink` (0..1) carries
    per-curve opacity; without it the lines are fully opaque."""
    if ink is None:
        ink = 1.0 - line_img.astype(np.float32) / 255.0
    return (flat.astype(np.float32) * (1.0 - ink)[..., None]).astype(np.uint8)


# ------------------------------------------------------ preview + report
def _sample(c, step=0.5):
    """Sample one primitive into polylines in pixel space, ~`step` px apart."""
    if isinstance(c[0], str):
        if c[0] == "line":
            p, q = np.asarray(c[1], float), np.asarray(c[2], float)
            n = max(2, int(np.hypot(*(q - p)) / step) + 1)
            u = np.linspace(0, 1, n)[:, None]
            return [p * (1 - u) + q * u]
        g = c[1]                                   # ellipse
        n = max(24, int(2 * np.pi * max(g["a"], g["b"]) / step))
        t = np.linspace(0, 2 * np.pi, n)
        co, si = np.cos(g["th"]), np.sin(g["th"])
        u, v = g["a"] * np.cos(t), g["b"] * np.sin(t)
        return [np.stack([g["h"] + u * co - v * si,
                          g["k"] + u * si + v * co], 1)]
    P = [np.asarray(p, float) for p in c]
    n = max(12, int(sum(np.hypot(*(P[i + 1] - P[i])) for i in range(3)) / step))
    u = np.linspace(0, 1, n)[:, None]
    w = 1 - u
    return [w ** 3 * P[0] + 3 * w * w * u * P[1] + 3 * w * u * u * P[2]
            + u ** 3 * P[3]]


def render_curves(curves, h, w, ss=2, thick=1):
    """Rasterise primitives (pixel coords) to a white canvas, anti-aliased."""
    canvas = np.full((h * ss, w * ss), 255, np.uint8)
    for c in curves:
        for poly in _sample(c):
            pts = np.round(poly * ss * 16).astype(np.int32).reshape(-1, 1, 2)
            cv2.polylines(canvas, [pts], False, 0, thick * ss,
                          cv2.LINE_AA, shift=4)
    return cv2.resize(canvas, (w, h), interpolation=cv2.INTER_AREA)


def fidelity(skeleton, curves, h, w):
    """How closely the drawn curves hug the detected centerlines (px)."""
    if not curves:
        return {}
    pts = np.vstack([q for c in curves for q in _sample(c, 0.5)])
    ys, xs = np.nonzero(skeleton)
    sk = np.stack([xs, ys], 1).astype(float)
    if len(sk) == 0 or len(pts) == 0:
        return {}
    d_cov = cKDTree(pts).query(sk)[0]              # skeleton -> curves
    d_pre = cKDTree(sk).query(pts)[0]              # curves  -> skeleton
    return {"coverage_%_within_1.5px": round(100 * float((d_cov <= 1.5).mean()), 2),
            "mean_dev_px": round(float(d_pre.mean()), 3),
            "p95_dev_px": round(float(np.percentile(d_pre, 95)), 3),
            "max_dev_px": round(float(d_pre.max()), 3)}


def run_pipeline(img, sens=1.5, sigma=1.2, nlm=5, merge=0, spur=8,
                 min_comp=14, min_len=10, fit_err=1.2, smooth=1.6,
                 max_segments=8000, verbose=True):
    """Full conversion; returns (curves_px, report dict, intermediates)."""
    h, w = img.shape[:2]
    rep, T = {"size": [w, h]}, time.time()

    def log(msg):
        if verbose:
            print(msg)

    edges, mag = detect_edges(img, sens, sigma, nlm, return_strength=True)
    rep["edge_px"] = int(edges.sum())
    rep["edge_density_%"] = round(float(100.0 * edges.mean()), 2)
    log(f"      edges      : {rep['edge_px']:,} px ({rep['edge_density_%']}% of image)")

    sk = centerline(edges, merge, min_comp)
    rep["skeleton_px"] = int(sk.sum())
    log(f"      centerline : {rep['skeleton_px']:,} px 1-px skeleton "
        f"(islands < {min_comp}px removed)")

    paths, st = trace_paths(sk, spur=spur)
    rep["paths_traced"] = len(paths)
    rep["spurs_pruned"] = st["spurs_pruned"]
    log(f"      trace      : {len(paths):,} paths, {st['spurs_pruned']} spurs pruned")

    fe = fit_err
    got = paths_to_curves(paths, fe, smooth, min_len)
    n_all = sum(len(p) for _, p in got)
    if n_all > max_segments:
        # Over budget.  Old behaviour (blur the fit, then drop the SHORTEST
        # paths) kept long faint texture and lost small strong detail.
        # Now every path gets  score = edge contrast * sqrt(length)  and the
        # best-scoring ones fill the budget.  Faint, short, wiggly paths -
        # the actual noise - are the first to go.
        scored = []
        for ln, pr in got:
            st = float(np.mean(curve_strength(pr, mag)))
            scored.append((st * np.sqrt(ln), ln, pr))
        scored.sort(key=lambda t: -t[0])
        keep, tot = [], 0
        for sc, ln, pr in scored:
            if tot + len(pr) > max_segments:
                continue
            keep.append((ln, pr))
            tot += len(pr)
        rep["paths_dropped_budget"] = len(got) - len(keep)
        log(f"      budget     : {n_all:,} primitives > {max_segments:,}; kept the "
            f"{len(keep):,} best-scoring paths, dropped {len(got) - len(keep):,}")
        got = keep
    rep["paths_kept"] = len(got)
    rep["paths_dropped_short"] = len(paths) - len(got)
    rep["fit_err_used_px"] = round(fe, 3)
    curves = [c for _, pr in got for c in pr]
    kinds = Counter("bezier" if not isinstance(c[0], str) else c[0] for c in curves)
    rep["primitives"] = len(curves)
    rep["primitive_kinds"] = dict(kinds)
    lens = np.array([ln for ln, _ in got]) if got else np.array([0.0])
    rep["path_length_px"] = {"median": round(float(np.median(lens)), 1),
                             "p90": round(float(np.percentile(lens, 90)), 1),
                             "longest": round(float(lens.max()), 1),
                             "total": round(float(lens.sum()), 0)}
    rep["fidelity"] = fidelity(sk, curves, h, w)
    rep["seconds"] = round(time.time() - T, 1)
    log(f"      fit        : {len(curves):,} primitives {dict(kinds)}")
    log(f"      fidelity   : {rep['fidelity']}")
    strength = curve_strength(curves, mag)
    rep["edge_contrast"] = {"median": round(float(np.median(strength)), 2) if len(strength) else 0,
                            "p90": round(float(np.percentile(strength, 90)), 2) if len(strength) else 0}
    return curves, rep, {"edges": edges, "skeleton": sk, "paths": paths,
                         "mag": mag, "strength": strength}


# ------------------------------------------------------- main
def main():
    ap = argparse.ArgumentParser(description="image -> Desmos art (v7: line art + colour fills)")
    ap.add_argument("image")
    ap.add_argument("--output", default="mega")
    ap.add_argument("--max-size", type=int, default=2000)
    ap.add_argument("--scale", type=float, default=1.0)
    # ---- line art
    ap.add_argument("--sens", type=float, default=1.5,
                    help="edge strictness: lower = more faint detail, higher = cleaner")
    ap.add_argument("--sigma", type=float, default=1.2,
                    help="edge scale (px): 1.2 finest detail, 2.0 calmest")
    ap.add_argument("--nlm", type=float, default=5, help="denoise strength; 0 off")
    ap.add_argument("--merge", type=float, default=0,
                    help="OPTIONAL: fill enclosed gaps up to this many px so a stroke's "
                         "double edge becomes one line (0 = off, best for dense art)")
    ap.add_argument("--spur", type=int, default=8, help="prune side-branches < px")
    ap.add_argument("--min-comp", type=int, default=14,
                    help="drop skeleton islands smaller than this many px")
    ap.add_argument("--min-len", type=int, default=10, help="drop paths shorter than px")
    ap.add_argument("--fit-err", type=float, default=1.2, help="max curve error px")
    ap.add_argument("--smooth", type=float, default=1.6, help="path smoothing sigma px")
    ap.add_argument("--no-conics", action="store_true")
    ap.add_argument("--max-segments", type=int, default=8000,
                    help="expression budget for the LINE ART (ignored when --target > 0)")
    ap.add_argument("--target", type=int, default=4000,
                    help="TOTAL expression budget (fills + lines); noise appears above "
                         "~4000.  0 = off, use --max-segments / --max-polys instead")
    ap.add_argument("--fill-share", type=float, default=0.30,
                    help="share of --target that colour fills may use")
    ap.add_argument("--max-polys", type=int, default=0,
                    help="cap on colour polygons (0 = automatic from --target)")
    ap.add_argument("--detail", type=float, default=3.5,
                    help="soft-detail pass (blush, shading): delta-E threshold; "
                         "lower = more patches, 0 = off")
    ap.add_argument("--max-detail", type=int, default=150,
                    help="cap on soft-detail patches")
    ap.add_argument("--line-width", type=float, default=1.5, help="Desmos line width")
    ap.add_argument("--line-opacity", type=float, default=None,
                    help="fixed opacity (0-1) for every line; default = adaptive "
                         "(by edge contrast in the image)")
    ap.add_argument("--line-opacity-min", type=float, default=0.30,
                    help="adaptive: opacity of the faintest edges")
    ap.add_argument("--line-opacity-max", type=float, default=0.90,
                    help="adaptive: opacity of the strongest edges")
    ap.add_argument("--line-opacity-gamma", type=float, default=0.8,
                    help="adaptive: <1 lifts mid-contrast lines, >1 fades them")
    # ---- colour
    ap.add_argument("--no-color", action="store_true", help="line art only")
    ap.add_argument("--K", type=int, default=14, help="number of flat colours")
    ap.add_argument("--min-area", type=int, default=150,
                    help="smallest colour region (px^2); lower = more polygons/detail")
    ap.add_argument("--ellipse-iou", type=float, default=0.90,
                    help="regions this ellipse-like become inequalities; lower = more")
    ap.add_argument("--max-ellipses", type=int, default=12)
    ap.add_argument("--no-folders", action="store_true",
                    help="plain expression list (fallback if folders fail to load)")
    args = ap.parse_args()

    global USE_CONICS
    USE_CONICS = not args.no_conics
    steps = 3 if args.no_color else 4

    print(f"[1/{steps}] load")
    img = load_image(args.image, args.max_size)
    h, w = img.shape[:2]
    print(f"      -> {w} x {h}")
    layers = crep = None
    seg_budget = args.max_segments
    if not args.no_color:
        print(f"[2/{steps}] colour fills: posterize + polygons + inequalities")
        mp = args.max_polys
        if args.target > 0 and not mp:
            mp = max(100, int(args.target * args.fill_share) - args.max_detail - 40)
        layers, crep, _ = build_color_layers(
            img, K=args.K, min_area=args.min_area, ellipse_iou=args.ellipse_iou,
            max_ellipses=args.max_ellipses, max_polys=mp, detail=args.detail,
            max_detail=args.max_detail)
        if args.target > 0:
            seg_budget = max(500, args.target - len(layers))
            print(f"      -> {len(layers):,} fill layers; line budget = {seg_budget:,}")
    elif args.target > 0:
        seg_budget = args.target
    print(f"[3/{steps}] line art: detect + trace + fit")
    curves_px, rep, inter = run_pipeline(
        img, args.sens, args.sigma, args.nlm, args.merge, args.spur,
        args.min_comp, args.min_len, args.fit_err, args.smooth, seg_budget)
    if args.line_opacity is not None:
        opac = [round(float(args.line_opacity), 2)] * len(curves_px)
    else:
        opac = line_opacities(inter["strength"], args.line_opacity_min,
                              args.line_opacity_max, args.line_opacity_gamma)
    lines, line_op = curves_to_desmos_op(
        curves_to_math(curves_px, h, w, args.scale), opac)
    if line_op:
        rep["line_opacity"] = {
            "mode": "fixed" if args.line_opacity is not None else "adaptive",
            "min": min(line_op), "median": float(np.median(line_op)), "max": max(line_op),
            "histogram_0.1_bins": np.histogram(line_op, bins=np.linspace(0, 1, 11))[0].tolist()}
    print(f"      -> {len(lines):,} line-art expressions")
    rep["line_art_expressions"] = len(lines)
    if args.no_color:
        print(f"[{steps}/{steps}] save")
        txt, js = save_outputs(lines, args.output, w, h, args.scale)
        rep["expressions"] = len(lines)
    else:
        fills = fills_to_expressions(layers, h, w, args.scale)
        print(f"[{steps}/{steps}] save")
        txt, js, total, extra = save_color_outputs(fills, lines, args.output, w, h,
                                            args.scale, f"{args.line_width:g}",
                                            folders=not args.no_folders,
                                            line_opacity=line_op)
        rep["color"] = crep
        rep["fill_expressions"] = len(fills)
        rep["expressions"] = total
        print(f"      -> {len(fills):,} fill + {len(lines):,} line = {total:,} expressions")
        print(f"      -> paste workflow: {txt}  +  {len(extra[0])} style chunk(s)  +  {extra[1]}")
    with open(args.output + "_report.json", "w") as f:
        json.dump(rep, f, indent=2, default=float)
    print(f"      -> {txt} ({os.path.getsize(txt) / 1e6:.2f} MB)")
    print(f"      -> {js} ({os.path.getsize(js) / 1e6:.2f} MB)")
    print(f"      -> {args.output}_report.json")


if __name__ == "__main__":
    main()


## 3 · Upload image

In [ ]:
from google.colab import files
uploaded = files.upload()
image_path = list(uploaded.keys())[0]
print(f'uploaded: {image_path}')

## 4 · Preview + detailed report

In [ ]:
import json
import numpy as np, cv2
import matplotlib.pyplot as plt
from desmos_mega import (load_image, run_pipeline, render_curves, build_color_layers,
                         compose_preview, line_opacities, render_ink)

MAX_SIZE = 2000                      # keep identical to the Generate cell
TARGET   = 4000                      # total expressions (fills + lines); keep identical to Generate
FILL_SHARE = 0.30                    # share of TARGET for colour fills
DETAIL, MAX_DETAIL = 3.5, 150        # soft-detail (blush) threshold dE / max patches. lower = more blush

img = load_image(image_path, MAX_SIZE)
h, w = img.shape[:2]

LO_MIN, LO_MAX, LO_GAMMA, FIXED = 0.30, 0.90, 0.8, None

max_polys = max(100, int(TARGET * FILL_SHARE) - MAX_DETAIL - 40)
layers, crep, cin = build_color_layers(img, K=14, min_area=150, ellipse_iou=0.90,
                                       max_polys=max_polys, detail=DETAIL, max_detail=MAX_DETAIL)
seg_budget = max(500, TARGET - len(layers))
curves, lrep, inter = run_pipeline(img, max_segments=seg_budget)
opac = [FIXED] * len(curves) if FIXED is not None else \
       line_opacities(inter["strength"], LO_MIN, LO_MAX, LO_GAMMA)
ink = render_ink(curves, opac, h, w)
final = compose_preview(cin["flat"], None, ink)

fig, ax = plt.subplots(1, 4, figsize=(24, 10))
ax[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); ax[0].set_title("source")
ax[1].imshow(1 - ink, cmap="gray", vmin=0, vmax=1)
ax[1].set_title(f"line art ({len(curves):,} expr, opacity {min(opac):.2f}-{max(opac):.2f})")
ax[2].imshow(cv2.cvtColor(cin["flat"], cv2.COLOR_BGR2RGB))
ax[2].set_title(f"colour fills - {len(layers)} layers (+{crep.get('detail_patches',0)} soft patches)")
ax[3].imshow(cv2.cvtColor(final, cv2.COLOR_BGR2RGB)); ax[3].set_title(f"result - {len(layers)+len(curves):,} expressions")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

print(json.dumps({"line_art": lrep, "color": crep}, indent=2, default=float))


## 5 · Generate
Creates, in draw order: `insane.txt` (equations), `insane_apply_styles.js` (console script), `insane_style_*.txt` (chunks), `insane_report.json`.

**Budget / quality flags** (add to the command below):
- `--target 4000` → TOTAL expressions (default). Raise for more detail (noise risk), `--target 0` = old behaviour
- `--fill-share 0.3` → share of the target that colour fills may use
- `--detail 3.5` → blush / soft-shading pass; **lower = more patches** (e.g. 2.5), `0` = off
- `--max-detail 150` → cap on those patches
- `--sens 2.0` → cleaner lines (default 1.5)

**Line opacity flags:** `--line-opacity-max 0.7`, `--line-opacity-min 0.2`, `--line-opacity 0.6`, `--line-width 1`. Use `--no-color` for line art only.

In [ ]:
!python desmos_mega.py "{image_path}" --max-size 2000 --target 4000 --detail 3.5 --output insane


## 6 · Download and load into Desmos
1. Open an **empty** graph at <https://www.desmos.com/calculator>.
2. Paste the contents of `insane.txt` into the first expression line. Wait until it finishes.
3. Press F12 → Console, paste the whole `insane_apply_styles.js`, press Enter. It prints `Applied styles to N expressions`.

If it warns that the expression count is wrong, a line didn't paste or the graph wasn't empty — styles are intentionally **not** applied then.

In [ ]:
import glob
from google.colab import files
for f in ['insane.txt', 'insane_apply_styles.js', 'insane_report.json'] + sorted(glob.glob('insane_style_*.txt')):
    files.download(f)
